# TippingPoint: Marketing Return Curves, Adstock & Multi-Channel Synergy
### *A Practical Guide to Media Saturation, Tipping Points, and Curve-Shifting (Performance vs. Brand Video)*

---

## 📖 Overview

In marketing mix modeling (MMM) and media planning, two critical questions determine budget success:
1. **At what spend level does an individual channel hit diminishing returns?**
2. **How does adding an upper-funnel / consideration channel shift the entire return curve up rather than just saturating lower-funnel performance channels?**

**TippingPoint** models non-linear diminishing returns using the **Hill Function** (the mathematical core of modern MMM methodologies like Google Meridian), calculates strategic inflection points, and supports carryover decay (Geometric and Weibull adstock).

---

### Key Concepts in this Tutorial:
* **Max Efficiency Point ($x_{\text{inflection}}$)**: The minimal marginal cost point where marginal ROAS (mROAS) peaks. Spend below this point is in the "warm-up" zone.
* **Max Profit Point ($x_{\text{diminishing}}$)**: The profitability floor where marginal ROAS drops to target mROAS ($mROAS = 1.0$). Spending beyond this point is unprofitable on the margin.
* **Optimal Scaling Zone**: The window between peak acquisition efficiency and diminishing returns.
* **Single vs. Multi-Channel Dynamics**: Why pouring more budget into saturated performance channels fails, and how Brand / Consideration video unlocks incremental response.

In [ ]:
# 1. Imports & Environment Setup
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure local package source is accessible
sys.path.insert(0, os.path.abspath("../src"))

from tippingpoint import (
    MarketingReturnCurve,
    MultiChannelMMM,
    PortfolioAllocator,
    weibull_adstock
)
from tippingpoint.math import hill_function, geometric_adstock

# Configure plot styling
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("TippingPoint imported successfully!")

## 📂 1. Data Import with Pandas (CSV Workflow)

In production, you will typically import campaign spend and revenue data from a CSV file exported from your attribution system, warehouse (e.g. BigQuery), or analytics database.

Below, we create a realistic dataset representing **104 weeks (2 years)** of marketing campaigns with two video strategies:
* **Performance Video (Direct Action / Demand Gen)**: Fast conversion cycles, short adstock carryover ($\approx 3$ days), sharp saturation ceiling.
* **Brand / Consideration Video (YouTube Reach / Connected TV)**: Longer-term awareness building, delayed peak and long carryover ($\approx 14$ days half-life), expands market size.
* **Organic Baseline**: Baseline demand from organic search, direct brand traffic, and repeat customers ($~\$15,000/\text{week}$).

In [ ]:
# Step 1: Generate synthetic historical data and save to CSV
np.random.seed(42)
weeks = 104
date_range = pd.date_range(start="2024-01-01", periods=weeks, freq="W")

# Spends per channel ($/week)
perf_spend = np.random.uniform(5000, 60000, size=weeks)
brand_spend = np.random.uniform(2000, 45000, size=weeks)
# Introduce seasonal pulses for brand campaigns
brand_spend[15:20] += 30000
brand_spend[45:52] += 40000

# Adstocked effective media exposure
# Performance Video: Fast decay (half-life ~ 3 days -> theta ~ 0.2)
perf_theta = 0.5 ** (1.0 / 3.0)
perf_adstocked = geometric_adstock(perf_spend, perf_theta)

# Brand Video: Weibull delayed peak & long carryover (shape=2.0, scale=4.0)
brand_adstocked = weibull_adstock(brand_spend, shape=2.0, scale=4.0, adstock_type="pdf")

# True Ground Truth Returns
baseline_revenue = 15000.0
perf_return_true = hill_function(perf_adstocked, beta=120000, alpha=1.6, K=25000)
brand_return_true = hill_function(brand_adstocked, beta=90000, alpha=1.2, K=30000)
noise = np.random.normal(0, 3000, size=weeks)

total_revenue = np.maximum(baseline_revenue + perf_return_true + brand_return_true + noise, 0)

# Build DataFrame and save to CSV
df_raw = pd.DataFrame({
    "date": date_range,
    "performance_video_spend": perf_spend,
    "brand_video_spend": brand_spend,
    "total_revenue": total_revenue
})

os.makedirs("data", exist_ok=True)
csv_file = "data/campaign_data.csv"
df_raw.to_csv(csv_file, index=False)
print(f"Sample data exported to: {csv_file}")

# --- USER IMPORT STEP ---
# This is where you import your own CSV:
df = pd.read_csv(csv_file, parse_dates=["date"])
df.head()

## 🎯 2. Single-Channel Response Curve: Performance Video

When analyzing a single media channel in isolation, `MarketingReturnCurve` provides a self-contained, lightweight model.

### 🛠️ Where You Can Influence Modeling Assumptions:

| Parameter / Assumption | What it Controls | When & How to Tune It |
| :--- | :--- | :--- |
| **Fitting Engine** | MLE (`from_historical_data`) vs. Bayesian MCMC (`fit_bayesian`) | Use MLE for instantaneous deterministic fits; use Bayesian to capture parameter uncertainty intervals ($90\%$ CI). |
| **`adstock_type`** | `'none'`, `'fixed'`, `'bounded'`, or `'free'` | If campaign decay is known (e.g. 3-day holdout study), use `adstock_type="fixed", adstock_fixed_days=3`. If uncertain, use `adstock_type="bounded", adstock_bounds=(1.0, 7.0)`. |
| **Priors (Bayesian)** | LogNormal prior on $(\beta, \alpha, K)$ | Set `priors={'beta': (mu, std), ...}` if historical MMM benchmarks or lift tests exist. |
| **`target_mroas`** | Threshold defining diminishing returns | Set to your minimum required incremental ROAS (e.g. `target_mroas=1.0` for breakeven, or `1.5` for margin target). |
| **`baseline`** | Organic / non-paid demand | Include baseline to avoid over-attributing organic search or direct sales to video ad spend. |

Let's fit the **Performance Video** channel below:

In [ ]:
# Extract channel spend and total return arrays
perf_spend_data = df["performance_video_spend"].values
revenue_data = df["total_revenue"].values

# Example: Specifying custom Bayesian Priors (optional user-influenced assumption)
# Prior format: (log_mean, standard_deviation)
custom_priors = {
    'beta': (np.log(150000), 0.5),  # Anticipated max saturation ceiling
    'alpha': (0.0, 0.4),            # S-curve power (alpha=1.0 is standard concave)
    'K': (np.log(30000), 0.5)       # Half-saturation spend estimate
}

# Fit Performance Video using Bayesian MCMC with Bounded Adstock (1 to 7 days half-life)
perf_model = MarketingReturnCurve.fit_bayesian(
    spend_array=perf_spend_data,
    return_array=revenue_data,
    channel_name="Performance_Video",
    priors=custom_priors,
    n_samples=1000,
    chains=4,
    burn_in=500,
    adstock_type="bounded",
    adstock_bounds=(1.0, 7.0),
    fit_baseline=True
)

# Display fitted summary
summary = perf_model.summary()
print("\n--- Performance Video Model Summary ---")
for key, val in summary["parameters"].items():
    print(f"  {key}: {val}")
print(f"Max Efficiency Point (Peak mROAS Spend): ${summary['tipping_points']['max_efficiency_point']:,.2f}")
print(f"Max Profit Point (mROAS = 1.0 Threshold): ${summary['tipping_points']['max_profit_point']:,.2f}")

In [ ]:
# Testing different budget scenarios on Performance Video
print("--- Scenario 1: Spending $5,000/week ---")
perf_model.evaluate_current_budget(current_spend=5000, target_mroas=1.0)

print("\n--- Scenario 2: Spending $35,000/week ---")
perf_model.evaluate_current_budget(current_spend=35000, target_mroas=1.0)

print("\n--- Scenario 3: Spending $80,000/week ---")
perf_model.evaluate_current_budget(current_spend=80000, target_mroas=1.0)

## 📊 3. Single-Channel Visualization

The visualizer plots:
1. **The Expected Incremental Return Curve** (solid blue) with **90% Posterior Credible Intervals** (shaded blue).
2. **Marginal ROAS Curve** (dashed gray) on the secondary axis.
3. **The Optimal Scaling Zone** (highlighted green band between peak efficiency and the diminishing returns point).
4. **Historical spend/return observations** (scatter dots).

In [ ]:
# Render the single-channel response curve with Tipping Points & Credible Intervals
fig = perf_model.plot_response_curve(
    target_mroas=1.0,
    current_spend=35000.0,
    show_intervals=True,
    scatter=(perf_model.adstock_spend(perf_spend_data), revenue_data),
    show=True
)

## 🚀 4. Adding a Second Channel: Brand Video & Synergistic Stacking

### ⚠️ The Single-Channel Performance Trap
When brands only invest in Performance Video (e.g. Demand Gen, bottom-of-funnel retargeting), they quickly exhaust in-market high-intent prospects. 
* As spend scales from $\$40\text{k} \to \$100\text{k}/\text{week}$, the performance response curve flattens dramatically.
* Marginal ROAS plunges below $1.0$, rendering additional performance investment unprofitable.

### 📈 How Brand / Consideration Video "Shifts the Curve Up"
Brand / Consideration Video (Connected TV, YouTube Reach, non-skippable video):
1. **Creates Future Demand**: Educates unprimed audiences who are not yet in-market.
2. **Has Longer Carryover**: Adstock effects last for weeks (captured via Weibull decay).
3. **Synergistic Stacking**: Instead of squeezing more out of a saturated performance channel, allocating budget into Brand Video stacks on top of the performance response curve, **shifting total customer acquisition potential upward**.

---

### Joint Multi-Channel Fitting (`MultiChannelMMM`)
To avoid double-counting returns between correlated channels, we fit both channels simultaneously:
$$Y_t = \text{Baseline} + \text{Hill}_{\text{perf}}\left(\text{Adstock}(S_{\text{perf}, t})\right) + \text{Hill}_{\text{brand}}\left(\text{Adstock}(S_{\text{brand}, t})\right) + \epsilon_t$$

In [ ]:
# Prepare multi-channel spend dictionary
spend_dict = {
    "Performance_Video": df["performance_video_spend"].values,
    "Brand_Video": df["brand_video_spend"].values
}

# Configure channel-specific adstock assumptions:
# Performance: short bounded adstock (1 to 5 days)
# Brand: longer bounded adstock (7 to 21 days)
adstock_types = {
    "Performance_Video": "bounded",
    "Brand_Video": "bounded"
}
adstock_bounds = {
    "Performance_Video": (1.0, 5.0),
    "Brand_Video": (7.0, 21.0)
}

# Fit joint MultiChannelMMM model using Bayesian MCMC
mmm = MultiChannelMMM.fit_bayesian(
    spend_data=spend_dict,
    return_array=revenue_data,
    n_samples=1000,
    chains=4,
    burn_in=500,
    fit_baseline=True,
    adstock_types=adstock_types,
    adstock_bounds=adstock_bounds
)

print(f"\nJoint Model Fitted Successfully!")
print(f"Estimated Organic Baseline Demand: ${mmm.baseline:,.2f} / week")
print("\nIndividual Channel Summaries:")
for cname, model in mmm.channels.items():
    s = model.summary()
    print(f"  [{cname}] Beta: ${s['parameters']['beta']:,.0f} | Half-Saturation (K): ${s['parameters']['K']:,.0f} | Half-Life: {s['parameters']['adstock_half_life_days']:.1f} days")

## 📈 5. Multi-Channel Visualization: Stacking & Shifting the Response Curve

Below, we visualize:
1. **Performance Video Alone** (blue curve) reaching its saturation ceiling.
2. **Performance Video + Brand Video Stacking** (stacked shaded curve) illustrating how introducing Brand Video breaks through the ceiling and shifts total customer revenue upward.
3. **Weekly Contribution Decomposition** over time.

In [ ]:
# Plotting Single vs Stacked Multi-Channel Curves
spend_range = np.linspace(0, 100000, 300)

perf_curve = mmm.channels["Performance_Video"]
brand_curve = mmm.channels["Brand_Video"]

perf_return = perf_curve.predict_incremental_return(spend_range)
# Assume Brand investment at 30% of spend range to illustrate synergistic scaling
brand_return_scaled = brand_curve.predict_incremental_return(spend_range * 0.5)

stacked_return = perf_return + brand_return_scaled

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Curve Shifting Effect
ax1.plot(spend_range, perf_return, color="#1a73e8", linewidth=2.5, label="Performance Video Alone")
ax1.plot(spend_range, stacked_return, color="#137333", linewidth=3, linestyle="-", label="Performance + Brand Video (Stacked)")
ax1.fill_between(spend_range, perf_return, stacked_return, color="#34a853", alpha=0.2, label="Added Lift from Brand Consideration")
ax1.axhline(perf_curve.beta, color="#d93025", linestyle=":", alpha=0.7, label=f"Performance Saturation Ceiling (${perf_curve.beta:,.0f})")

ax1.set_title("How Brand Video Shifts the Response Curve Up", fontsize=13, fontweight="bold")
ax1.set_xlabel("Weekly Spend ($)", fontsize=11)
ax1.set_ylabel("Incremental Return ($)", fontsize=11)
ax1.legend(loc="upper left", frameon=True, facecolor="white", framealpha=0.9)
ax1.yaxis.set_major_formatter('${x:,.0f}')
ax1.xaxis.set_major_formatter('${x:,.0f}')

# Subplot 2: Marginal ROAS Comparison
mroas_perf = perf_curve.predict_marginal_return(spend_range)
mroas_brand = brand_curve.predict_marginal_return(spend_range)

ax2.plot(spend_range, mroas_perf, color="#1a73e8", linewidth=2, label="mROAS: Performance Video")
ax2.plot(spend_range, mroas_brand, color="#f29900", linewidth=2, label="mROAS: Brand Video")
ax2.axhline(1.0, color="#d93025", linestyle="--", label="Profitability Floor (mROAS = 1.0)")

ax2.set_title("Marginal ROAS by Channel", fontsize=13, fontweight="bold")
ax2.set_xlabel("Weekly Spend ($)", fontsize=11)
ax2.set_ylabel("Marginal ROAS ($ Return / $ Spend)", fontsize=11)
ax2.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.9)
ax2.xaxis.set_major_formatter('${x:,.0f}')
ax2.set_ylim(0, 3.5)

plt.tight_layout()
plt.show()

In [ ]:
# Time-Series Revenue Attribution Decomposition
contributions = mmm.predict_channel_contributions(spend_dict)

fig, ax = plt.subplots(figsize=(14, 6))
time_axis = df["date"]

baseline_series = np.full(len(df), contributions["Baseline"])
perf_series = contributions["Performance_Video"]
brand_series = contributions["Brand_Video"]

ax.stackplot(
    time_axis,
    baseline_series,
    perf_series,
    brand_series,
    labels=["Organic Baseline", "Performance Video Contribution", "Brand Video Contribution"],
    colors=["#dadce0", "#4285f4", "#ea4335"],
    alpha=0.85
)
ax.plot(time_axis, df["total_revenue"], color="#202124", linewidth=1.5, label="Observed Total Revenue")

ax.set_title("Historical Revenue Decomposition Across Video Channels", fontsize=14, fontweight="bold")
ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Weekly Revenue ($)", fontsize=11)
ax.yaxis.set_major_formatter('${x:,.0f}')
ax.legend(loc="upper left", frameon=True, facecolor="white", framealpha=0.95)
plt.tight_layout()
plt.show()

## ⚖️ 6. Portfolio Optimization: Finding the Optimal Budget Split

Now that both channels are modeled with non-linear diminishing returns, what is the **globally optimal spend split** for a target weekly budget (e.g. $\$80,000/\text{week}$)?

The `PortfolioAllocator` solves:
$$\max_{s_1, s_2} \sum_{i} \text{Hill}_i(s_i) \quad \text{subject to} \quad \sum s_i = \text{Total Budget}, \quad \text{min\_spend}_i \le s_i \le \text{max\_spend}_i$$

It equalizes **marginal ROAS across channels** ($mROAS_{\text{perf}} = mROAS_{\text{brand}}$), moving budget away from the saturated performance tail into higher-efficiency brand consideration.

In [ ]:
# Obtain allocator directly from the fitted multi-channel model
allocator = mmm.get_allocator()

total_budget = 80000.0  # $80,000 / week
channel_bounds = {
    "Performance_Video": (10000.0, 60000.0),
    "Brand_Video": (5000.0, 45000.0)
}

allocation_result = allocator.allocate_budget(
    total_budget=total_budget,
    channel_bounds=channel_bounds
)

print(f"=== Optimal Budget Allocation for ${total_budget:,.2f} / week ===")
print(f"Expected Incremental Media Return: ${allocation_result['expected_total_return']:,.2f}")
print(f"Overall Blended ROAS: {allocation_result['overall_roas']:.2f}x\n")

for cname, spend in allocation_result["allocation"].items():
    mroas_at_alloc = allocation_result["marginal_roas"][cname]
    ret = allocation_result["channel_returns"][cname]
    print(f"  • {cname}: ${spend:,.2f} ({spend/total_budget*100:.1f}%) | Return: ${ret:,.2f} | Marginal ROAS: {mroas_at_alloc:.2f}")

## 📋 7. Summary & Practitioner Best Practices

### 💡 Key Takeaways:
1. **Single-Channel Saturation is Inevitable**:
   Always check your channel's `max_profit_point` ($x_{\text{diminishing}}$). Spending past this threshold delivers sub-target marginal returns.
2. **Channel Synergy Unlocks New Ceilings**:
   Brand / Consideration channels expand the top of the funnel. Introducing brand awareness creates carryover that shifts the entire performance response curve upwards.
3. **Equalize Marginal ROAS**:
   When allocating multi-channel budgets, don't allocate by average ROAS. Use `PortfolioAllocator` to equalize **marginal ROAS**, ensuring every additional marketing dollar is deployed to its highest-yield channel.
4. **Calibrate with Incrementality Experiments**:
   If you have geo-lift or holdout experiments, pass them to `calibration_experiments=[{"spend": ..., "lift": ..., "se": ...}]` during Bayesian fitting to anchor observational MMM to causal truth.

---
*Created with [TippingPoint](https://github.com/RyanAugust/TippingPoint).*